# EDA gold layer

### Check if number of rows seems reasonable

In [0]:
%sql
SELECT COUNT(*) FROM marathos.gold.fct_results; -- 6261875
SELECT COUNT(*) FROM marathos.gold.dim_event; -- 76320
SELECT COUNT(*) FROM marathos.gold.dim_athlete; -- 3251151
SELECT COUNT(*) FROM marathos.gold.dim_date; -- 5785

### Check out tables

In [0]:
%sql
USE CATALOG marathos;

USE SCHEMA gold;

SELECT *
FROM fct_results;

SELECT *
FROM dim_event;

SELECT *
FROM dim_athlete;

SELECT *
FROM dim_date;


In [0]:
%sql
SELECT 
  COUNT(*) AS total,
  COUNT(performance_seconds) AS seconds_count,
  COUNT(performance_km) AS km_count
FROM marathos.gold.fct_results

### Check fastest and slowest time per event_type

In [0]:
%sql
SELECT e.event_type, MIN(f.performance_seconds), MAX(f.performance_seconds)
FROM marathos.gold.fct_results f
LEFT JOIN marathos.gold.dim_event e ON e.event_id = f.event_id
WHERE f.performance_seconds > 0
GROUP BY e.event_type;

### Check average speed per country (top 20)

In [0]:
SELECT a.country_name, ROUND(AVG(f.average_speed), 2) AS avg_speed
FROM marathos.gold.fct_results f
JOIN marathos.gold.dim_athlete a USING (athlete_id)
GROUP BY a.country_name
ORDER BY avg_speed DESC
LIMIT 20;

### Check number of finishers per year and event_type

In [0]:
SELECT e.event_year, e.event_type, COUNT(*) AS num_results
FROM marathos.gold.fct_results f
JOIN marathos.gold.dim_event e USING (event_id)
GROUP BY e.event_year, e.event_type
ORDER BY e.event_year DESC;

### Check gender distribution on event_type

In [0]:
SELECT e.event_type, a.gender, COUNT(*) AS num_athletes
FROM marathos.gold.fct_results f
JOIN marathos.gold.dim_athlete a USING (athlete_id)
JOIN marathos.gold.dim_event e USING (event_id)
GROUP BY e.event_type, a.gender;

In [0]:
%sql
SELECT 
  COUNT(*) AS total,
  COUNT(performance_seconds) AS seconds_count,
  COUNT(performance_km) AS km_count,
  SUM(CASE WHEN performance_seconds = 0 THEN 1 ELSE 0 END) AS zero_seconds
FROM marathos.gold.fct_results

## Check views

In [0]:
SELECT *
FROM marathos.gold.mart_time_events;

In [0]:
SELECT *
FROM marathos.gold.mart_distance_events;

### Mart for Swedish runners for events between 2010 to 2022
Decided to delete it - to slim for steakholders

In [0]:
USE CATALOG marathos;
USE SCHEMA gold;
SELECT 
a.athlete_id,
a.age_category,
a.birth_year,
a.club,
a.gender,
e.event_name,
e.event_type,
e.event_year,
e.start_date,
e.end_date,
e.distance,
f.performance_seconds,
f.performance_km,
f.average_speed
FROM dim_athlete a
LEFT JOIN fct_results f ON a.athlete_id = f.athlete_id
LEFT JOIN dim_event e ON f.event_id = e.event_id
WHERE a.country_name ='Sweden'
AND e.event_year BETWEEN 2010 AND 2022
ORDER BY e.event_year;

In [0]:
SELECT 
DISTINCT club 
FROM marathos.gold.dim_athlete;


In [0]:
SELECT 
  a.age_category,
  COUNT(*) AS total_rows,
  AVG(f.performance_seconds / 3600.0) AS avg_performance_hours
FROM marathos.gold.dim_athlete a
JOIN marathos.gold.fct_results f 
  ON a.athlete_id = f.athlete_id
GROUP BY a.age_category
ORDER BY a.age_category ASC;

## Stockholm Trail Classic (SWE)
Verify that STockholm Trail Classic (SWE) went through the obt-pipeline and into the gold layer 

In [0]:
SELECT DISTINCT 'Event name'
FROM marathos.bronze.raw_marathos
WHERE 'Event name' LIKE '%Stockholm%'

In [0]:
SELECT DISTINCT event_name
FROM marathos.silver.marathos_obt
WHERE event_name LIKE '%Stockholm%'

In [0]:
SELECT * 
FROM marathos.gold.dim_event
WHERE event_name = 'Stockholm Trail Classic (SWE)'

In [0]:
SELECT *
FROM marathos.gold.fct_results f
JOIN marathos.gold.dim_event e ON f.event_id = e.event_id
WHERE e.event_name = 'Stockholm Trail Classic (SWE)'
LIMIT 5